# B.1 — 打包 InternImage 環境（conda-pack）→ Kaggle dataset

**目的**：HuBMAP 是 code competition，計分時用 hidden test 重跑 notebook。要讓 InternImage
(mmdet3.x) 與作者模型(mmdet2.x) 在「同一個 notebook run」裡都跑到 hidden test，必須把
InternImage 的整個 Python 環境打包成 self-contained tarball，之後在作者 notebook 用 subprocess 跑。

**這個 build notebook**：
- 網路 **ON**（只產環境 tarball，不是 submission）
- 用 Kaggle GPU 的 nvcc 把 mmcv 編成 **sm_75**（T4）
- 產出 `/kaggle/working/iienv.tar.gz`（~3-4GB）→ Save Version 後把 Output 存成 dataset `internimage-env`

**只需跑一次**。


In [ ]:
import torch, subprocess
print('base torch:', torch.__version__, '| cuda:', torch.cuda.is_available())
print(subprocess.run(['nvcc','--version'],capture_output=True,text=True).stdout.strip().splitlines()[-1])
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none')


In [ ]:
# 建乾淨 conda env `ii`（py3.10）+ torch 2.1.2 cu121（self-contained）
!conda create -n ii python=3.10 -y
!conda run -n ii pip install -q torch==2.1.2 torchvision==0.16.2 --index-url https://download.pytorch.org/whl/cu121
!conda run -n ii pip install -q setuptools wheel ninja numpy
!conda run -n ii python -c "import torch;print('ii torch:',torch.__version__,torch.version.cuda)"


In [ ]:
# source-build mmcv 2.1.0（對齊 ii torch ABI + sm_75）。git tag 才會真 source build。
import os, subprocess
env = dict(os.environ)
env['MMCV_WITH_OPS']='1'; env['FORCE_CUDA']='1'; env['TORCH_CUDA_ARCH_LIST']='7.5'
env.setdefault('CUDA_HOME','/usr/local/cuda')
ret = subprocess.run(
    'conda run -n ii pip install -v --no-build-isolation '
    '"mmcv @ git+https://github.com/open-mmlab/mmcv.git@v2.1.0"',
    shell=True, env=env)
assert ret.returncode==0, 'mmcv source build 失敗'
print('✓ mmcv 2.1.0 source build 完成')


In [ ]:
# mmdet 3.3.0 + mmpretrain + 推理套件
!conda run -n ii pip install -q mmdet==3.3.0 mmpretrain==1.2.0 "albumentations>=1.3,<1.4" pycocotools opencv-python-headless shapely terminaltables
!conda run -n ii python -c "import torch,mmcv,mmdet; from mmcv import ops; print('ii ok:',torch.__version__,mmcv.__version__,mmdet.__version__)"


In [ ]:
# conda-pack → tarball
!conda install -n base -c conda-forge conda-pack -y
!conda pack -n ii -o /kaggle/working/iienv.tar.gz --force
import os; print('tarball size: %.2f GB' % (os.path.getsize('/kaggle/working/iienv.tar.gz')/1024**3))


In [ ]:
# 驗證：解開 → conda-unpack → 用 packed python import（模擬 fusion notebook 用法）
!rm -rf /tmp/iitest && mkdir -p /tmp/iitest
!tar xzf /kaggle/working/iienv.tar.gz -C /tmp/iitest
!/tmp/iitest/bin/conda-unpack
!/tmp/iitest/bin/python -c "import torch,mmcv,mmdet; from mmcv import ops; print('UNPACKED OK:',torch.__version__,'| cuda:',torch.cuda.is_available(),'| mmcv:',mmcv.__version__,'| mmdet:',mmdet.__version__)"
print('\n印出 UNPACKED OK 且 cuda: True → env 可用。Save Version 後把 iienv.tar.gz 存成 dataset internimage-env。')
